# ChopChop + IterGen's unconstrained completion step

**Runtime: G4 GPU.** Then **Runtime > Run all**. Nothing to edit. About 20-30 minutes.

The IterGen integration finishes a function body that is still open when IterGen stops
(greedy, no constraints, at most 128 tokens); the ChopChop integration has no such step.
This applies the same step to the two finished ChopChop runs on Drive
(`tools/chopchop_fallback.py`). Only the tasks with an open body are touched (about 100 per model).

Output: `final/<model>/chopchop_lim600_fallback128_<commit>/` on Drive, with `fallback.jsonl`.
The runtime disconnects itself at the end.

In [ ]:
REPO_URL    = 'https://github.com/BowenX307/constrained-llm-js-codegen-final.git'
REPO_COMMIT = '31981d35ee9c11a131443ee82fbaafa46a32366b'
EXPECT_GPU  = 'RTX PRO 6000'      # G4, the same GPU type as every final run
MODELS      = ['Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen3.5-4B']
SOURCE_RUN  = 'chopchop_lim600_2c14eef'
AUTO_DISCONNECT = True

In [ ]:
import os, sys, shutil, subprocess, glob
REPO, WORK = '/content/final', '/content/work'
DRIVE_ROOT = '/content/drive/MyDrive/constrained-llm-js-codegen'
smi = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = smi.stdout.strip().splitlines()[0] if smi.returncode == 0 and smi.stdout.strip() else ''
print('GPU', GPU_NAME or 'none')
if EXPECT_GPU not in GPU_NAME:
    raise RuntimeError(f'This runtime has {GPU_NAME or "no GPU"}; use G4 (Runtime > Change runtime type).')

from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir(f'{DRIVE_ROOT}/final'), 'Drive folder constrained-llm-js-codegen/final not found'

if os.path.isdir(REPO):
    shutil.rmtree(REPO)
for cmd in (['git', 'init', '-q', REPO], ['git', '-C', REPO, 'remote', 'add', 'origin', REPO_URL],
            ['git', '-C', REPO, 'fetch', '-q', '--depth', '1', 'origin', REPO_COMMIT],
            ['git', '-C', REPO, 'checkout', '-q', 'FETCH_HEAD']):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + ' failed:\n' + r.stderr)
print('code at', REPO_COMMIT[:7])

def pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('pip install failed: ' + ' '.join(a) + '\n' + r.stderr[-2000:])
from importlib.metadata import version
if version('transformers') != '5.20.1':
    was = version('transformers')
    pip('transformers==5.20.1')
    if 'transformers' in sys.modules:
        raise RuntimeError(f'transformers was {was}; installed 5.20.1. Runtime > Restart session, then Run all.')
pip('lark', 'regex', 'networkx', 'fire', 'interegular', 'tqdm', 'datasets', 'jsonschema', 'psutil')
if subprocess.run([sys.executable, '-c', 'from syncode.parsers.itergen_parser import IGParser'],
                  cwd=REPO, capture_output=True).returncode != 0:
    pip('--no-deps', '--ignore-requires-python', '--force-reinstall',
        'git+https://github.com/structuredllm/syncode.git@4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4')
ENV = dict(os.environ, PYTHONPATH=REPO, PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false',
           HF_HUB_DISABLE_PROGRESS_BARS='1')
print('ready')

In [ ]:
try:
    for model in MODELS:
        slug = model.split('/')[-1]
        src_drive = f'{DRIVE_ROOT}/final/{slug}/{SOURCE_RUN}'
        src, out = f'{WORK}/{slug}/{SOURCE_RUN}', f'{WORK}/{slug}/fallback'
        out_drive = f'{DRIVE_ROOT}/final/{slug}/chopchop_lim600_fallback128_{REPO_COMMIT[:7]}'
        os.makedirs(src, exist_ok=True)
        for f in glob.glob(f'{src_drive}/*.json'):
            shutil.copy(f, src)
        print(f'== {slug}: {len(glob.glob(src + "/mbpp_*.json"))} tasks copied from Drive', flush=True)
        r = subprocess.run([sys.executable, f'{REPO}/tools/chopchop_fallback.py', '--model', model,
                            '--run', src, '--out', out], cwd=REPO, env=ENV, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print('\n'.join(l for l in r.stdout.splitlines()
                        if not any(s in l for s in ('Warning', 'warn(', 'Loading weights', 'torch_dtype'))))
        if r.returncode != 0:
            raise RuntimeError(f'{slug}: chopchop_fallback.py failed')
        shutil.copytree(out, out_drive, dirs_exist_ok=True)
        print(f'saved to {out_drive}', flush=True)
    print('ALL DONE')
finally:
    if AUTO_DISCONNECT:
        import time
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        time.sleep(30)
        runtime.unassign()